# RAG Concepts — Step by Step

## Why RAG?

An LLM only knows what it saw during training. It has **never seen your shop's handbook**, so if you ask it about your refund policy it will either refuse or, worse, **make up** a confident answer (a _hallucination_).

**RAG (Retrieval-Augmented Generation)** fixes this without retraining the model:

| Word           | Meaning                                                      |
| -------------- | ------------------------------------------------------------ |
| **Retrieval**  | Find the parts of your documents that relate to the question |
| **Augmented**  | Paste those parts into the prompt as _context_               |
| **Generation** | The LLM writes an answer **from that context**               |

It's like an **open-book exam**: the model doesn't need to memorise the handbook, it just needs the right page in front of it.

## The two phases

```
INDEXING (once, before any questions)
  handbook.md ──► Load ──► Chunk ──► Embed ──► Store (vector store)

QUERYING (every question)
  question ──► Embed ──► Retrieve top-k chunks ──► Augment prompt ──► LLM ──► answer
```

**Series:** **RAG-01 Concepts (this one)** → RAG-02 Scores → RAG-03 Qdrant


## Step 0 — Setup

**Prerequisites:** Ollama running, with `gemma3:1b` and `nomic-embed-text` pulled, and `shop_handbook.md` next to this notebook.


In [ ]:
# uv add --active langchain langchain-text-splitters langchain-ollama

In [ ]:
DATA_PATH = "shop_handbook.md"

CHAT_PROVIDER = "ollama"
CHAT_MODEL = "gemma3:1b"
EMBEDDING_PROVIDER = "ollama"
EMBEDDING_MODEL = "nomic-embed-text:latest"
TOP_K = 3

SYSTEM_PROMPT = (
    "You are a helpful Clause Shop assistant. "
    "Answer only from the provided context. "
    "If the answer is not in the context, say you don't know."
)

# Part 1 — Indexing

## Step 1 — Load

LangChain represents every piece of text as a **`Document`** with two fields:

- **`page_content`**: the text itself
- **`metadata`**: a dictionary of facts _about_ the text (source file, page number, ...). It travels with the text through every step, which is how we can later say _where_ an answer came from.

For a `.md` or `.txt` file we don't need a special loader: read the file and wrap it in a `Document`.


In [ ]:
from pathlib import Path
from langchain_core.documents import Document


text = Path(DATA_PATH).read_text(encoding="utf-8")
doc = Document(page_content=text, metadata={"source": DATA_PATH})

print("Characters:", len(doc.page_content))
print("Metadata:  ", doc.metadata)
print("\n--- First 300 characters ---\n" + doc.page_content[:300])

### Document loaders (LangChain, Nov 2026)

A **document loader** turns a file (PDF, Word, web page, ...) into `Document` objects. Which one you use depends on how messy the file is.

| Loader                                       | Package                        | Use it for                                                                                                  | Status                                                                                                                                        |
| -------------------------------------------- | ------------------------------ | ----------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------- |
| `Path.read_text` + `Document`                | `langchain-core`               | `.txt`, `.md` (this notebook)                                                                               | ✅ No loader needed                                                                                                                           |
| `pypdf` + `Document`                         | `pypdf`                        | Simple text PDFs. LangChain's own tutorial now does this                                                    | ✅ Current                                                                                                                                    |
| `PyPDFLoader`, `TextLoader`, `PyMuPDFLoader` | `langchain-community`          | Older tutorials                                                                                             | ⚠️ **Deprecated**: the package was sunset and its repo archived (June 2026). It still imports, with a `DeprecationWarning`, but gets no fixes |
| `PyMuPDF4LLMLoader`                          | `langchain-pymupdf4llm`        | PDF → Markdown, fast and local                                                                              | ✅ Standalone package                                                                                                                         |
| `DoclingLoader`                              | `langchain-docling`            | Complex PDFs with tables and layouts, plus DOCX/PPTX. Open source and local, but heavier (downloads models) | ✅ Standalone package                                                                                                                         |
| `UnstructuredLoader`                         | `langchain-unstructured`       | Many file types (PDF, DOCX, HTML, email), locally or through their API                                      | ✅ Standalone package                                                                                                                         |
| LlamaParse                                   | `llama-cloud` (LlamaIndex SDK) | Hardest documents: scans, charts, messy layouts                                                             | ☁️ Cloud service with an API key. No LangChain loader: wrap its output in `Document` yourself                                                 |

**Rule of thumb:** start with the simplest option that gives clean text. Bad loading means bad chunks, and no later step can fix that.


## Step 2 — Chunk

Why not embed the whole handbook as one piece?

- **Precision:** one embedding for the whole file blends every topic together, so a refund question would match "everything a bit".
- **Context size:** we only want to send the LLM the few paragraphs that matter, not the whole document.

So we split it into **chunks**:

- **`chunk_size=600`**: each chunk has at most about 600 characters.
- **`chunk_overlap=100`**: neighbouring chunks share about 100 characters, so a sentence cut at a border still appears whole in one of them.
- **`separators`**: try to split at paragraphs (`\n\n`) first, then lines, then words, and only as a last resort in the middle of a word. That's what _Recursive_ means.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)
chunks = text_splitter.split_documents([doc])

print(f"1 document -> {len(chunks)} chunks")
for i, c in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ({len(c.page_content)} chars) | metadata: {c.metadata} ---")
    print(c.page_content[:200])

Notice that every chunk **inherited the metadata** (`source`) from the original document.

**Trade-off:** chunks that are too small lose meaning ("14 days" without saying 14 days _for what_). Chunks that are too large mix several topics. 300–1000 characters is a common starting range.


## Step 3 — Embed

An **embedding model** turns text into a list of numbers (a **vector**) that captures its _meaning_. Texts with similar meaning get vectors that point in similar directions, even when they share no words.


In [ ]:
from langchain.embeddings import init_embeddings


embedd = init_embeddings(model=EMBEDDING_MODEL, provider=EMBEDDING_PROVIDER)

vector = embedd.embed_query("How do I get a refund?")
print("Vector length:", len(vector))          # 768 for nomic-embed-text
print("First 5 numbers:", [round(float(x), 4) for x in vector[:5]])

### Meaning, not keywords

**Cosine similarity** measures how closely two vectors point in the same direction. Higher means more similar in meaning.


In [ ]:
import math


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


q = embedd.embed_query("How do I get a refund?")

for sentence in ["Can I get my money back?", "Customers can return items within 14 days.", "What's the weather in Lahore?"]:
    print(f"{cosine(q, embedd.embed_query(sentence)):.3f}  {sentence}")

"Can I get my money back?" shares **no words** with "How do I get a refund?", but it should score high. The weather question should score lowest.

That is why RAG uses embeddings instead of keyword search.


## Step 4 — Store

A **vector store** holds every chunk together with its vector, and can quickly find the vectors closest to a new one.

`from_documents` embeds **every chunk** and stores the results.


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore


vector_store = InMemoryVectorStore.from_documents(documents=chunks, embedding=embedd)
print(f"Stored {len(chunks)} chunks")

`InMemoryVectorStore` lives in **RAM**, so everything is lost when the program stops and the handbook is embedded again on every run. That's fine for learning. In **RAG-03** we replace it with Qdrant, which saves to disk.


# Part 2 — Querying

## Step 5 — Retrieve

The question is embedded with the **same model** that embedded the chunks. The store then returns the **top-k** closest chunks (`k=3`).


In [ ]:
query = "How many days do I have to request a refund?"

retrieved_docs = vector_store.similarity_search(query, k=TOP_K)

for rank, d in enumerate(retrieved_docs, start=1):
    print(f"--- Rank {rank} ---")
    print(d.page_content[:200], "\n")

**Two rules:**

- Use the **same embedding model** for the question and the chunks. Vectors from different models can't be compared.
- `similarity_search` **always** returns `k` chunks, even when none of them is relevant. RAG-02 deals with that by adding scores.


## Step 6 — Augment

This is the **"A" in RAG**. We join the retrieved chunks into a `context` string and put it into the prompt next to the question.

Print the prompt to see exactly what the LLM receives.


In [ ]:
context = "\n\n".join(d.page_content for d in retrieved_docs)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"},
]

print(messages[1]["content"])

- The **system prompt** sets the rules: _answer only from the context, otherwise say you don't know_. That's our main defence against hallucination.
- The **user message** carries both the context and the question.


## Step 7 — Generate

`temperature=0.1` keeps the answers factual and consistent, with less creative guessing.


In [ ]:
from langchain.chat_models import init_chat_model


llm = init_chat_model(model=CHAT_MODEL, model_provider=CHAT_PROVIDER, temperature=0.1)

print("AI:", llm.invoke(messages).text)

## Step 8 — Proof: without RAG vs with RAG

Ask the **same question twice**: once with no context, and once with the retrieved context.


In [ ]:
question = "What is the refund window for electronics at Clause Shop?"


# Without RAG: the LLM only has its training data
print("WITHOUT RAG:", llm.invoke(question).text[:300], "\n")


# With RAG: retrieve, augment, generate
ctx = "\n\n".join(d.page_content for d in vector_store.similarity_search(question, k=TOP_K))
rag_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Context: {ctx}\n\nQuestion: {question}"},
]
print("WITH RAG:   ", llm.invoke(rag_messages).text)

Without RAG, the model has to guess, so it gives a generic or invented answer. With RAG, it answers from **your** handbook.

That difference is the whole point of RAG.


## Step 9 — Everything together

This is the full querying phase in one function, the same logic as the script. Run the cell and type a question.


In [ ]:
def ask(query):
    retrieved_docs = vector_store.similarity_search(query, k=TOP_K)             # Retrieve
    context = "\n\n".join(d.page_content for d in retrieved_docs)               # Augment
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"},
    ]
    return llm.invoke(messages).text                                      # Generate


query = input("You: ").strip()
print("AI:", ask(query))

## Summary

| Step        | Phase    | What happens                                  | Code                             |
| ----------- | -------- | --------------------------------------------- | -------------------------------- |
| 1. Load     | Indexing | File → `Document` (text + metadata)           | `Path.read_text`, `Document`     |
| 2. Chunk    | Indexing | Split into small, overlapping pieces          | `RecursiveCharacterTextSplitter` |
| 3. Embed    | Indexing | Text → vector of meaning                      | `init_embeddings`                |
| 4. Store    | Indexing | Save the vectors so they can be searched      | `InMemoryVectorStore`            |
| 5. Retrieve | Querying | Find the top-k chunks closest to the question | `similarity_search`              |
| 6. Augment  | Querying | Put the chunks into the prompt                | `messages`                       |
| 7. Generate | Querying | The LLM answers from the context              | `llm.invoke`                     |

**Limits of this version, solved in the next notebooks:**

- We can't tell whether the retrieved chunks are actually relevant → **RAG-02:** `similarity_search_with_score`
- Vectors are lost when the program stops → **RAG-03:** Qdrant
